# 🧪 Praktikum 08 – Agenten-Architekturen: ReAct & Reflection
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Einen autonomen Loop mit State-Management bauen · Komplexe Tool-Aufrufe mit Argumenten parsen · Self-Reflection zur Fehlerkorrektur einsetzen

⏱️ **Dauer:** 90 Minuten

In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und fuehre die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Der Plan-and-Execute Agent ⏱️ 30 min
Statt direkt zu antworten, erstellt der Agent erst einen Plan.

In [ ]:
def get_plan(task):
    prompt = f"Erstelle einen 3-Schritte-Plan, um diese Aufgabe zu lösen: {task}\nAntworte nur als Liste."
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 160},
        keep_alive="10m",
    )
    return response["message"]["content"]

task = "Recherchiere das Wetter in Berlin, berechne die Kleidungsempfehlung und schreibe eine E-Mail."
print("PLAN:\n", get_plan(task))

## Teil 2 – ReAct Loop mit Memory ⏱️ 40 min
Wir bauen ein Gedächtnis ein, damit der Agent weiß, was er bereits getan hat.

In [ ]:
import ast


def safe_eval_expr(expression):
    allowed_binary_ops = {
        ast.Add: lambda left, right: left + right,
        ast.Sub: lambda left, right: left - right,
        ast.Mult: lambda left, right: left * right,
        ast.Div: lambda left, right: left / right,
        ast.FloorDiv: lambda left, right: left // right,
        ast.Mod: lambda left, right: left % right,
        ast.Pow: lambda left, right: left ** right,
    }
    allowed_unary_ops = {
        ast.UAdd: lambda value: value,
        ast.USub: lambda value: -value,
    }

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_binary_ops:
            return allowed_binary_ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_unary_ops:
            return allowed_unary_ops[type(node.op)](_eval(node.operand))
        raise RuntimeError(f"Unsupported calculator expression: {expression}")

    syntax_tree = ast.parse(expression, mode="eval")
    return _eval(syntax_tree)


class SimpleAgent:
    def __init__(self, model):
        self.model = model
        self.memory = []
        self.tools = {"calc": lambda expression: str(safe_eval_expr(expression))}
        self.system_prompt = (
            "Du bist ein einfacher ReAct-Agent. Ohne OBSERVATION darfst du nur genau eine Zeile "
            "im Format ACTION: calc(<vollstaendiger arithmetischer Ausdruck>) ausgeben. "
            "Sobald eine OBSERVATION vorliegt, darfst du nur noch genau eine Zeile im Format "
            "FINAL: <Ergebnis> ausgeben."
        )

    def run(self, query):
        self.memory = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": query},
        ]

        for _ in range(3):
            response = ollama.chat(
                model=self.model,
                messages=self.memory,
                think=False,
                options={"temperature": 0, "num_predict": 40, "stop": ["\n"]},
                keep_alive="10m",
            )["message"]["content"].strip()
            if not response:
                raise RuntimeError("Model returned an empty response.")

            print(f"Agent: {response}")
            self.memory.append({"role": "assistant", "content": response})

            action_match = re.fullmatch(r"ACTION: (\w+)\((.*?)\)", response)
            if action_match:
                tool_name, argument = action_match.groups()
                if tool_name not in self.tools:
                    raise RuntimeError(f"Unknown tool requested by model: {tool_name}")

                observation = self.tools[tool_name](argument)
                print(f"Tool Output: {observation}")
                self.memory.append({"role": "user", "content": f"OBSERVATION: {observation}"})
                continue

            final_match = re.fullmatch(r"FINAL: (.+)", response)
            if final_match:
                print(f"Final Answer: {final_match.group(1)}")
                return final_match.group(1)

            raise RuntimeError(f"Unexpected agent response: {response}")

        raise RuntimeError("Agent did not finish within 3 iterations.")


agent = SimpleAgent(MODEL)
agent.run("Berechne 15 * 15 und dann plus 10.")

## Teil 3 – Reflection (Self-Criticism) ⏱️ 20 min
Der Agent soll seine eigene Antwort bewerten und ggf. korrigieren.

In [ ]:
def reflect_and_fix(answer):
    prompt = f"Kritisiere die folgende Antwort auf logische Fehler und korrigiere sie falls nötig: {answer}"
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 160},
        keep_alive="10m",
    )
    return response["message"]["content"]

wrong_ans = "Ein Kilo Federn ist schwerer als ein Kilo Blei, weil Federn mehr Volumen haben."
print("Korrektur:\n", reflect_and_fix(wrong_ans))

## 🧩 Aufgaben
1. **Argument Parsing:** Erweitere den Agenten so, dass er Tools mit zwei Argumenten (z.B. `add(a, b)`) versteht.
2. **State Export:** Speichere die `self.memory` Liste nach einem Run als JSON-Datei ab. Warum ist das für Debugging hilfreich?
3. **Stop-Kriterium:** Implementiere ein Limit für die Anzahl der Tool-Aufrufe, um Endlosschleifen zu verhindern.